# Análise Exploratória de Dados — Classificação de Raças de Gatos

Este notebook contém a análise exploratória completa do projeto, dividida em duas partes:
- **Parte A**: Análise do dataset cru (antes do pipeline de pré-processamento)
- **Parte B**: Análise das features extraídas (após o pipeline)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import cv2
import os
from pathlib import Path
from scipy import stats
from collections import Counter

# Configurações globais de visualização
plt.rcParams["figure.dpi"]       = 150
plt.rcParams["font.family"]      = "DejaVu Sans"
plt.rcParams["axes.spines.top"]  = False
plt.rcParams["axes.spines.right"]= False
sns.set_palette("tab10")

# Paths
DATASET_PATH  = Path("../data/oxford-iiit-pet")
IMAGES_PATH   = DATASET_PATH / "images"
ANNOT_PATH    = DATASET_PATH / "annotations"
TRIMAPS_PATH  = ANNOT_PATH   / "trimaps"
OUTPUT_PATH   = Path("../outputs/visuals/eda")
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

SEED = 42
np.random.seed(SEED)

# 12 raças de gato do Oxford-IIIT Pet
CAT_BREEDS = [
    "Abyssinian", "Bengal", "Birman", "Bombay", "British_Shorthair",
    "Egyptian_Mau", "Maine_Coon", "Persian", "Ragdoll",
    "Russian_Blue", "Siamese", "Sphynx"
]

---

## PARTE A — ANÁLISE DO DATASET CRU

## 1. Inventário do Dataset

O Oxford-IIIT Pet Dataset contém imagens de 37 raças de animais domésticos. Para este trabalho, filtramos apenas as 12 raças de gatos (species_id == 1). Nesta seção verificamos a integridade e distribuição dos dados antes de qualquer pré-processamento.

In [ ]:
def load_cat_dataframe(annot_path: Path, images_path: Path) -> pd.DataFrame:
    """
    Carrega e filtra apenas imagens de gatos do Oxford-IIIT Pet.
    Retorna DataFrame com colunas:
    [image_name, breed_name, breed_id, species_id, split, image_path, exists, file_size_kb]
    """
    rows = []
    for split_file, split_name in [("trainval.txt", "train"), ("test.txt", "test")]:
        filepath = annot_path / split_file
        with open(filepath) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 4:
                    continue
                name, class_id, species_id, breed_id = parts[0], int(parts[1]), int(parts[2]), int(parts[3])
                if species_id != 1:   # 1 = gato
                    continue
                breed_name = "_".join(name.split("_")[:-1])
                img_path   = images_path / f"{name}.jpg"
                rows.append({
                    "image_name": name,
                    "breed_name": breed_name,
                    "breed_id":   breed_id - 1,   # 0-indexed
                    "split":      split_name,
                    "image_path": img_path,
                    "exists":     img_path.exists(),
                    "file_size_kb": round(img_path.stat().st_size / 1024, 1) if img_path.exists() else 0
                })
    return pd.DataFrame(rows)

df = load_cat_dataframe(ANNOT_PATH, IMAGES_PATH)

print(f"Total de imagens de gato: {len(df)}")
print(f"Imagens encontradas:      {df['exists'].sum()}")
print(f"Imagens ausentes:         {(~df['exists']).sum()}")
print(f"\nRaças únicas encontradas: {df['breed_name'].nunique()}")
print(f"\nDistribuição treino/teste:")
print(df['split'].value_counts())

In [ ]:
# Tabela cruzada raça × split
pivot = df.groupby(["breed_name", "split"]).size().unstack(fill_value=0)
pivot["total"] = pivot.sum(axis=1)
pivot = pivot.sort_values("total", ascending=False)

print("\nDistribuição por raça:")
print(pivot.to_string())

# Salvar como CSV para o artigo
pivot.to_csv(OUTPUT_PATH / "breed_distribution.csv")

## 2. Balanceamento das Classes

Um dataset desbalanceado pode enviesar o classificador. Verificamos aqui se todas as 12 raças têm representação similar, o que influencia diretamente a escolha da métrica de avaliação (acurácia vs F1 macro).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: total por raça
breed_counts = df.groupby("breed_name").size().sort_values(ascending=False)
axes[0].bar(range(len(breed_counts)), breed_counts.values,
            color=sns.color_palette("tab10", len(breed_counts)))
axes[0].set_xticks(range(len(breed_counts)))
axes[0].set_xticklabels(breed_counts.index, rotation=45, ha="right", fontsize=9)
axes[0].set_title("Total de imagens por raça")
axes[0].set_ylabel("Nº de imagens")
axes[0].axhline(breed_counts.mean(), color="red", linestyle="--",
                label=f"Média: {breed_counts.mean():.0f}")
axes[0].legend()

# Gráfico 2: treino vs teste empilhado
pivot_plot = df.groupby(["breed_name", "split"]).size().unstack(fill_value=0)
pivot_plot = pivot_plot.loc[breed_counts.index]
pivot_plot.plot(kind="bar", stacked=True, ax=axes[1],
                color=["#4C72B0", "#DD8452"])
axes[1].set_title("Distribuição treino/teste por raça")
axes[1].set_ylabel("Nº de imagens")
axes[1].set_xticklabels(pivot_plot.index, rotation=45, ha="right", fontsize=9)
axes[1].legend(title="Split")

plt.suptitle("Balanceamento do Dataset — Raças de Gato", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(OUTPUT_PATH / "class_balance.png", bbox_inches="tight")
plt.show()

# Estatísticas de balanceamento
cv_balance = breed_counts.std() / breed_counts.mean()
print(f"Coeficiente de variação entre classes: {cv_balance:.3f}")
print("→ CV < 0.2 indica dataset bem balanceado")

## 3. Galeria Visual por Raça

Antes de processar as imagens, é fundamental visualizar amostras de cada raça para entender as variações de cor, textura, pose e iluminação presentes no dataset. Essa análise justifica as etapas de normalização e equalização de histograma no pipeline.

In [ ]:
def load_rgb(path: Path) -> np.ndarray:
    """Carrega imagem como RGB via OpenCV."""
    img = cv2.imread(str(path))
    if img is None:
        return None
    return img[:, :, ::-1]   # BGR → RGB

# Grade: 12 raças × 4 exemplos por raça
n_breeds   = len(CAT_BREEDS)
n_examples = 4
fig, axes  = plt.subplots(n_breeds, n_examples, figsize=(14, 42))

for i, breed in enumerate(CAT_BREEDS):
    breed_df = df[(df["breed_name"] == breed) & df["exists"]].sample(
        min(n_examples, len(df[df["breed_name"] == breed])),
        random_state=SEED
    )
    for j, (_, row) in enumerate(breed_df.iterrows()):
        img = load_rgb(row["image_path"])
        if img is not None:
            axes[i, j].imshow(img)
        axes[i, j].axis("off")
        if j == 0:
            axes[i, j].set_title(breed.replace("_", " "),
                                  fontsize=9, fontweight="bold", loc="left")

plt.suptitle("Galeria Visual — 4 amostras por raça", fontsize=14, fontweight="bold", y=1.001)
plt.tight_layout()
plt.savefig(OUTPUT_PATH / "breed_gallery.png", bbox_inches="tight")
plt.show()

print("Figura salva: breed_gallery.png")

## 4. Dimensões e Tamanho das Imagens

As imagens do dataset têm dimensões variadas — essa análise justifica a necessidade de redimensionamento para 128×128 pixels antes da extração de features, garantindo vetores de tamanho fixo para o classificador.

In [ ]:
widths, heights, aspects = [], [], []
sample_df = df[df["exists"]].sample(min(500, len(df)), random_state=SEED)

for _, row in sample_df.iterrows():
    img = cv2.imread(str(row["image_path"]))
    if img is not None:
        h, w = img.shape[:2]
        heights.append(h)
        widths.append(w)
        aspects.append(w / h)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(widths,   bins=30, color="#4C72B0", edgecolor="white")
axes[0].set_title("Distribuição de Larguras")
axes[0].set_xlabel("Pixels")
axes[0].axvline(np.mean(widths), color="red", linestyle="--",
                label=f"Média: {np.mean(widths):.0f}px")
axes[0].legend()

axes[1].hist(heights,  bins=30, color="#DD8452", edgecolor="white")
axes[1].set_title("Distribuição de Alturas")
axes[1].set_xlabel("Pixels")
axes[1].axvline(np.mean(heights), color="red", linestyle="--",
                label=f"Média: {np.mean(heights):.0f}px")
axes[1].legend()

axes[2].hist(aspects,  bins=30, color="#55A868", edgecolor="white")
axes[2].set_title("Distribuição de Aspect Ratio (W/H)")
axes[2].set_xlabel("Razão W/H")
axes[2].axvline(1.0, color="gray", linestyle=":", label="Quadrado (1.0)")
axes[2].legend()

plt.suptitle("Dimensões das Imagens (amostra de 500)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(OUTPUT_PATH / "image_dimensions.png", bbox_inches="tight")
plt.show()

print(f"Largura  — média: {np.mean(widths):.0f}px | min: {min(widths)}px | max: {max(widths)}px")
print(f"Altura   — média: {np.mean(heights):.0f}px | min: {min(heights)}px | max: {max(heights)}px")
print(f"Aspect   — média: {np.mean(aspects):.3f} | std: {np.std(aspects):.3f}")
print(f"Tamanho médio dos arquivos: {df['file_size_kb'].mean():.1f} KB")

## 5. Análise de Cor por Raça

Cada raça possui características cromáticas distintas — Bombay é predominantemente preta, Siamese tem padrão bicolor, Russian Blue tem pelagem acinzentada. Analisar os histogramas de cor por raça revela o poder discriminativo desse tipo de feature, justificando o uso de histogramas HSV no vetor de características.

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(15, 18))
axes = axes.flatten()
colors = ["#E74C3C", "#2ECC71", "#3498DB"]   # R, G, B
channel_names = ["R", "G", "B"]

for i, breed in enumerate(CAT_BREEDS):
    breed_imgs = df[(df["breed_name"] == breed) & df["exists"]].sample(
        min(30, len(df[df["breed_name"] == breed])), random_state=SEED
    )
    hist_acc = [np.zeros(256) for _ in range(3)]
    for _, row in breed_imgs.iterrows():
        img = load_rgb(row["image_path"])
        if img is not None:
            img_resized = cv2.resize(img, (128, 128))
            for c in range(3):
                hist, _ = np.histogram(img_resized[:, :, c], bins=256, range=(0, 255))
                hist_acc[c] += hist
    # Normalizar
    for c in range(3):
        hist_acc[c] = hist_acc[c] / hist_acc[c].sum()
        axes[i].plot(hist_acc[c], color=colors[c], alpha=0.8,
                     linewidth=1.2, label=channel_names[c])
    axes[i].set_title(breed.replace("_", " "), fontsize=10, fontweight="bold")
    axes[i].set_xlim([0, 255])
    axes[i].set_xlabel("Intensidade")
    axes[i].set_ylabel("Frequência norm.")
    if i == 0:
        axes[i].legend(loc="upper left")

plt.suptitle("Histograma RGB Médio por Raça (30 amostras/raça)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(OUTPUT_PATH / "rgb_histograms_per_breed.png", bbox_inches="tight")
plt.show()

In [ ]:
brightness_data = []
for _, row in df[df["exists"]].iterrows():
    img = load_rgb(row["image_path"])
    if img is not None:
        gray = 0.299 * img[:,:,0] + 0.587 * img[:,:,1] + 0.114 * img[:,:,2]
        brightness_data.append({
            "breed_name": row["breed_name"],
            "mean_brightness": gray.mean(),
            "std_brightness":  gray.std()
        })

bright_df = pd.DataFrame(brightness_data)
breed_order = bright_df.groupby("breed_name")["mean_brightness"].median().sort_values().index

fig, ax = plt.subplots(figsize=(13, 5))
sns.boxplot(data=bright_df, x="breed_name", y="mean_brightness",
            order=breed_order, palette="viridis", ax=ax)
ax.set_xticklabels([b.replace("_", " ") for b in breed_order],
                   rotation=45, ha="right")
ax.set_title("Distribuição de Brilho Médio por Raça", fontsize=12, fontweight="bold")
ax.set_xlabel("Raça")
ax.set_ylabel("Brilho médio (0–255)")
plt.tight_layout()
plt.savefig(OUTPUT_PATH / "brightness_per_breed.png", bbox_inches="tight")
plt.show()

print("Raças mais escuras (menor brilho médio):")
print(bright_df.groupby("breed_name")["mean_brightness"].median().sort_values().head(3))
print("\nRaças mais claras (maior brilho médio):")
print(bright_df.groupby("breed_name")["mean_brightness"].median().sort_values().tail(3))

## 6. Análise de Textura

A variância local de intensidade é um proxy simples para textura — raças com pelagem longa (Persian, Maine Coon) tendem a ter alta variância local, enquanto raças de pelagem curta (Bombay, Siamese) apresentam superfícies mais uniformes. Essa diferença justifica o uso de LBP (Local Binary Pattern) no vetor de features.

In [ ]:
texture_data = []
for _, row in df[df["exists"]].sample(min(600, len(df)), random_state=SEED).iterrows():
    img = load_rgb(row["image_path"])
    if img is not None:
        img_r = cv2.resize(img, (128, 128))
        gray  = 0.299*img_r[:,:,0] + 0.587*img_r[:,:,1] + 0.114*img_r[:,:,2]
        # Variância local como proxy de textura
        local_var = np.array([
            gray[i-1:i+2, j-1:j+2].var()
            for i in range(1, 127) for j in range(1, 127)
        ]).mean()
        texture_data.append({
            "breed_name":  row["breed_name"],
            "local_var":   local_var
        })

tex_df = pd.DataFrame(texture_data)
breed_order_tex = tex_df.groupby("breed_name")["local_var"].median().sort_values().index

fig, ax = plt.subplots(figsize=(13, 5))
sns.boxplot(data=tex_df, x="breed_name", y="local_var",
            order=breed_order_tex, palette="rocket", ax=ax)
ax.set_xticklabels([b.replace("_", " ") for b in breed_order_tex],
                   rotation=45, ha="right")
ax.set_title("Variância Local de Textura por Raça", fontsize=12, fontweight="bold")
ax.set_xlabel("Raça")
ax.set_ylabel("Variância local média")
plt.tight_layout()
plt.savefig(OUTPUT_PATH / "texture_variance_per_breed.png", bbox_inches="tight")
plt.show()

## 7. Análise das Trimaps

O Oxford-IIIT Pet fornece trimaps pixel a pixel indicando foreground (gato), background e região ambígua. Analisamos a proporção média de foreground por raça, o que indica o quanto do frame é ocupado pelo animal — raças com animais menores no frame são mais desafiadoras para segmentação.

In [ ]:
trimap_data = []
for _, row in df[df["exists"]].iterrows():
    trimap_path = TRIMAPS_PATH / f"{row['image_name']}.png"
    if trimap_path.exists():
        trimap = cv2.imread(str(trimap_path), cv2.IMREAD_GRAYSCALE)
        if trimap is not None:
            total      = trimap.size
            foreground = np.sum(trimap == 1)   # valor 1 = animal
            background = np.sum(trimap == 2)
            ambiguous  = np.sum(trimap == 3)
            trimap_data.append({
                "breed_name":   row["breed_name"],
                "fg_ratio":     foreground / total,
                "bg_ratio":     background / total,
                "amb_ratio":    ambiguous  / total
            })

tri_df = pd.DataFrame(trimap_data)
breed_order_fg = tri_df.groupby("breed_name")["fg_ratio"].median().sort_values().index

fig, ax = plt.subplots(figsize=(13, 5))
sns.boxplot(data=tri_df, x="breed_name", y="fg_ratio",
            order=breed_order_fg, palette="mako", ax=ax)
ax.set_xticklabels([b.replace("_", " ") for b in breed_order_fg],
                   rotation=45, ha="right")
ax.set_title("Proporção de Foreground (animal) nas Trimaps por Raça",
             fontsize=12, fontweight="bold")
ax.set_xlabel("Raça")
ax.set_ylabel("Razão foreground (0–1)")
ax.axhline(tri_df["fg_ratio"].mean(), color="red", linestyle="--",
           label=f"Média geral: {tri_df['fg_ratio'].mean():.2f}")
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_PATH / "trimap_fg_ratio.png", bbox_inches="tight")
plt.show()

print(f"Proporção média de foreground: {tri_df['fg_ratio'].mean():.3f}")
print(f"Raça com menor área de animal: {breed_order_fg[0]}")
print(f"Raça com maior área de animal: {breed_order_fg[-1]}")

---

## PARTE B — ANÁLISE DAS FEATURES EXTRAÍDAS

**Nota**: Esta parte do notebook deve ser executada **após** o pipeline completo com `python src/pipeline.py`!

## 8. Inspeção do Vetor de Features

Após o pipeline de pré-processamento, cada imagem é representada por um vetor de 99 features. Aqui verificamos a integridade do vetor e inspecionamos as distribuições estatísticas de cada bloco de features.

In [ ]:
data   = np.load("../outputs/features/features.npz", allow_pickle=True)
X      = data["X"]         # (N, 99)
y      = data["y"]         # (N,)
breeds = data["breeds"]    # (12,)
split  = data["split"]     # (N,)

print(f"Shape de X:          {X.shape}")
print(f"Shape de y:          {y.shape}")
print(f"Raças:               {list(breeds)}")
print(f"Splits disponíveis:  {np.unique(split)}")
print(f"\nValores nulos:       {np.isnan(X).sum()}")
print(f"Valores infinitos:   {np.isinf(X).sum()}")
print(f"\nDistribuição de classes:")
for i, b in enumerate(breeds):
    print(f"  {b:25s}: {np.sum(y == i):3d} imagens")

# Nomes dos blocos de features
feature_names = (
    ["mean", "std", "entropy", "skewness", "kurtosis"] +       # 5 estatísticas
    [f"hsv_H_{i}" for i in range(16)] +                         # 16 H
    [f"hsv_S_{i}" for i in range(16)] +                         # 16 S
    [f"hsv_V_{i}" for i in range(16)] +                         # 16 V
    ["area", "perimeter", "compactness", "aspect_ratio"] +      # 4 forma
    [f"dct_{i}" for i in range(10)] +                           # 10 DCT
    [f"lbp_{i}" for i in range(32)]                             # 32 LBP
)

assert len(feature_names) == 99

## 9. Variância das Features

Features com baixa variância entre classes têm pouco poder discriminativo. Identificamos aqui quais features mais contribuem para separar as raças, o que pode orientar uma futura seleção de features.

In [ ]:
# Variância de cada feature
feature_vars = X.var(axis=0)
feat_df = pd.DataFrame({
    "feature": feature_names,
    "variance": feature_vars
}).sort_values("variance", ascending=False)

fig, ax = plt.subplots(figsize=(15, 4))
colors_feat = ["#4C72B0" if v > np.percentile(feature_vars, 75)
               else "#DD8452" if v > np.percentile(feature_vars, 25)
               else "#C0C0C0" for v in feat_df["variance"]]
ax.bar(range(99), feat_df["variance"], color=colors_feat, width=0.8)
ax.set_title("Variância de Cada Feature (ordenado por variância)",
             fontsize=12, fontweight="bold")
ax.set_xlabel("Feature (índice ordenado)")
ax.set_ylabel("Variância")
ax.axhline(np.percentile(feature_vars, 25), color="red", linestyle=":",
           label="Percentil 25")
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_PATH / "feature_variance.png", bbox_inches="tight")
plt.show()

print("Top 10 features de maior variância:")
print(feat_df.head(10)[["feature", "variance"]].to_string(index=False))
print("\nTop 10 features de menor variância (candidatas a remoção):")
print(feat_df.tail(10)[["feature", "variance"]].to_string(index=False))

## 10. Correlação entre Features

Alta correlação entre features indica redundância no vetor — features muito correlacionadas carregam a mesma informação e podem prejudicar alguns classificadores. O SVM com kernel RBF é relativamente robusto a isso, mas é importante documentar para o artigo.

In [ ]:
corr_matrix = np.corrcoef(X.T)   # (99, 99)

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))   # triângulo superior
sns.heatmap(corr_matrix, mask=mask, cmap="RdBu_r",
            center=0, vmin=-1, vmax=1,
            xticklabels=False, yticklabels=False,
            ax=ax, cbar_kws={"shrink": 0.7})
ax.set_title("Matriz de Correlação entre Features (99×99)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(OUTPUT_PATH / "feature_correlation.png", bbox_inches="tight")
plt.show()

# Pares altamente correlacionados
high_corr = []
for i in range(99):
    for j in range(i+1, 99):
        if abs(corr_matrix[i, j]) > 0.90:
            high_corr.append((feature_names[i], feature_names[j],
                               round(corr_matrix[i, j], 3)))

print(f"Pares com |correlação| > 0.90: {len(high_corr)}")
if high_corr:
    for a, b, r in high_corr[:10]:
        print(f"  {a:20s} × {b:20s} → r = {r}")

## 11. Análise de Componentes Principais (PCA)

A PCA reduz o vetor de 99 features para 2 dimensões, permitindo visualizar se as raças formam clusters separáveis no espaço de features. Clusters bem definidos indicam que o vetor de features tem alto poder discriminativo, antecipando boa performance do classificador.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=2, random_state=SEED)
X_pca = pca.fit_transform(X_scaled)

print(f"Variância explicada — PC1: {pca.explained_variance_ratio_[0]:.3f}")
print(f"Variância explicada — PC2: {pca.explained_variance_ratio_[1]:.3f}")
print(f"Total: {pca.explained_variance_ratio_.sum():.3f}")

fig, ax = plt.subplots(figsize=(12, 9))
palette = sns.color_palette("tab10", len(breeds)) + sns.color_palette("Set2", 2)

for i, breed in enumerate(breeds):
    mask_breed = y == i
    ax.scatter(X_pca[mask_breed, 0], X_pca[mask_breed, 1],
               label=breed.replace("_", " "), alpha=0.5,
               s=20, color=palette[i])

ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var.)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var.)")
ax.set_title("PCA 2D das Features — Separabilidade por Raça",
             fontsize=12, fontweight="bold")
ax.legend(loc="upper right", fontsize=8, markerscale=2,
          bbox_to_anchor=(1.18, 1))
plt.tight_layout()
plt.savefig(OUTPUT_PATH / "pca_2d.png", bbox_inches="tight")
plt.show()

In [ ]:
pca_full = PCA(random_state=SEED).fit(X_scaled)
cumvar   = np.cumsum(pca_full.explained_variance_ratio_)
n_95 = np.argmax(cumvar >= 0.95) + 1
n_99 = np.argmax(cumvar >= 0.99) + 1

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(1, len(cumvar)+1), cumvar, color="#4C72B0", linewidth=2)
ax.axhline(0.95, color="red",   linestyle="--", label=f"95% ({n_95} comps.)")
ax.axhline(0.99, color="orange",linestyle="--", label=f"99% ({n_99} comps.)")
ax.fill_between(range(1, len(cumvar)+1), cumvar, alpha=0.1, color="#4C72B0")
ax.set_xlabel("Número de Componentes")
ax.set_ylabel("Variância Acumulada Explicada")
ax.set_title("Curva de Variância Acumulada (PCA)", fontsize=12, fontweight="bold")
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_PATH / "pca_cumvar.png", bbox_inches="tight")
plt.show()

print(f"Componentes para explicar 95% da variância: {n_95}")
print(f"Componentes para explicar 99% da variância: {n_99}")

## 12. Features Discriminativas por Raça

Visualizamos as features mais importantes individualmente por raça. Isso permite identificar quais características o classificador provavelmente usará para distinguir as raças — e quais raças são mais parecidas entre si.

In [ ]:
# Features de maior variância entre as médias por classe (poder discriminativo)
class_means = np.array([X[y == i].mean(axis=0) for i in range(len(breeds))])
between_var  = class_means.var(axis=0)
top_features_idx = np.argsort(between_var)[::-1][:6]

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for ax_i, feat_idx in enumerate(top_features_idx):
    data_feat = [X[y == i, feat_idx] for i in range(len(breeds))]
    bp = axes[ax_i].boxplot(data_feat, patch_artist=True, notch=False)
    colors_bp = sns.color_palette("tab10", len(breeds))
    for patch, color in zip(bp["boxes"], colors_bp):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    axes[ax_i].set_xticks(range(1, len(breeds)+1))
    axes[ax_i].set_xticklabels([b.replace("_", " ") for b in breeds],
                                rotation=45, ha="right", fontsize=7)
    axes[ax_i].set_title(f"Feature: {feature_names[feat_idx]}", fontsize=10, fontweight="bold")
    axes[ax_i].set_ylabel("Valor")

plt.suptitle("Top 6 Features com Maior Poder Discriminativo entre Raças",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(OUTPUT_PATH / "top_features_boxplot.png", bbox_inches="tight")
plt.show()

## 13. Sumário Final para o Artigo

In [ ]:
print("=" * 60)
print("SUMÁRIO DA ANÁLISE EXPLORATÓRIA")
print("=" * 60)
print(f"\nDataset:")
print(f"  Total de imagens de gato: {len(df)}")
print(f"  Raças analisadas:         {len(CAT_BREEDS)}")
print(f"  Imagens de treino:        {(split == 'train').sum()}")
print(f"  Imagens de teste:         {(split == 'test').sum()}")
print(f"\nFeatures:")
print(f"  Dimensão do vetor:        {X.shape[1]}")
print(f"  Valores nulos:            {np.isnan(X).sum()}")
print(f"  Pares correlac. (>0.9):   {len(high_corr)}")
print(f"\nPCA:")
print(f"  PCs para 95% variância:   {n_95}")
print(f"  PCs para 99% variância:   {n_99}")
print(f"\nFiguras salvas em: {OUTPUT_PATH}")
saved = list(OUTPUT_PATH.glob("*.png"))
for f in sorted(saved):
    print(f"  {f.name}")